# Volume 2. FSM And PFA Utilities

Question: how do the package-level automata helpers expose discrete computation over assembly states?

This notebook demonstrates the maintained `FSMNetwork` and `PFANetwork` APIs. It does not reconstruct every condition in the theoretical Turing-completeness results.

In [ ]:
from collections import Counter

from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import FSMNetwork, PFANetwork, overlap

First build a deterministic parity machine. The state toggles each time it reads `1`.

In [ ]:
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")

fsm = FSMNetwork(
    brain,
    states=["even", "odd"],
    symbols=["1"],
    transitions=[("even", "1", "odd"), ("odd", "1", "even")],
    initial_state="even",
    n=4_000,
    k=60,
    beta=0.05,
    rounds=6,
)

print("initial:", fsm.current_state)
print("trajectory:", fsm.run(["1", "1", "1", "1"]))

The state lexicon is inspectable. The two state assemblies should be much less similar than an assembly compared with itself.

In [ ]:
even = fsm.state_lexicon["even"]
odd = fsm.state_lexicon["odd"]
print("overlap(even, even):", overlap(even, even))
print("overlap(even, odd):", f"{overlap(even, odd):.3f}")

Now build a probabilistic automaton. Reset before each sample when you want independent draws from the same initial state.

In [ ]:
pfa = PFANetwork(
    brain,
    states=["start", "left", "right"],
    symbols=["go"],
    transitions=[("start", "go", "left", 0.5), ("start", "go", "right", 0.5)],
    initial_state="start",
    n=4_000,
    k=60,
    beta=0.05,
    rounds=6,
)

counts = Counter()
for seed in range(12):
    pfa.reset()
    counts[pfa.step("go", seed=seed)] += 1

print(dict(counts))

What to notice: the API gives you readable state names and trajectories. The underlying implementation still uses assemblies internally, so you can inspect overlaps when debugging.